# Eurex API FAQ & Special Cases

[![Open In Colab](https://img.shields.io/badge/Google%20Colab-F9AB00?style=for-the-badge&logo=googlecolab&logoColor=white)](https://colab.research.google.com/github/ViktorHD/eurex-api/blob/main/notebooks/FAQ_cases.ipynb) 
[![Binder](https://img.shields.io/badge/Binder-579ACA?style=for-the-badge&logo=binder&logoColor=white)](https://mybinder.org/v2/gh/ViktorHD/eurex-api/main?labpath=notebooks%2FFAQ_cases.ipynb) 
[![Databricks](https://img.shields.io/badge/Databricks-FF3621?style=for-the-badge&logo=databricks&logoColor=white)](https://community.cloud.databricks.com/?o=0#external-import?github_url=https%3A%2F%2Fgithub.com%2FViktorHD%2Feurex-api%2Fblob%2Fmain%2Fnotebooks%2FFAQ_cases.ipynb)

This notebook covers concrete, highly requested examples and recipes for processing reference data from the [Eurex GraphQL API](https://www.eurex.com/ex-en/data/free-reference-data-api).

## Case 1: Breaking Down Product-Level MinLotSize to Single Contract Level

### Problem Description
Within the Eurex reference data model, block trade thresholds (`MinLotSize`) from `TESProfiles` are defined on the **product** level, depending on whether the instrument is standard or flexible. For standard/simple instruments, the minimum lot size can vary based on how far out the expiry date is.

Specifically, `TESProfiles` has a `MinExpiryRange` field, which indicates that a particular `MinLotSize` is valid *until (inclusive)* that `MinExpiryRange` value is reached. `MinExpiryRange` represents the expiration index (i.e. the chronological sequence of contract expirations, starting from `1` for the front month/closest expiry).

To determine the actual `MinLotSize` for a **single individual contract** (at the contract ID/ISIN level), we need to:
1. **Fetch Expirations:** For a given product, query `Expirations` to retrieve the mapping between a `MasterContract` and its chronological `ExpirationIndex` (along with `ProductID` and `ExpirationDate`).
2. **Fetch TESProfiles:** Query `TESProfiles` to get the `MinLotSize` and `MinExpiryRange` thresholds for standard simple instruments (`InstrumentType: "SIMPLE_INSTRUMENT"` and `TESType: "BLOCK"`).
3. **Fetch Contracts:** Query `Contracts` to obtain individual contracts for that product.
4. **Align and Map:** Join the Contracts and Expirations on `ProductID` and `MasterContract` to associate an `ExpirationIndex` with each single contract. Then, map that index to the corresponding `MinLotSize` from the product's `TESProfiles` where `ExpirationIndex <= MinExpiryRange` (selecting the most specific, i.e. smallest, valid range).

### 1. Setup & API Helper

We start by importing the necessary libraries and establishing a connection to the Eurex GraphQL API using the public demo key.

In [0]:
import sys
!{sys.executable} -m pip install pandas requests
import requests
import json
import pandas as pd

API_URL = "https://api.developer.deutsche-boerse.com/eurex-prod-graphql/"
# Public demo key from: https://www.eurex.com/ex-en/data/free-reference-data-api
API_KEY = "68cdafd2-c5c1-49be-8558-37244ab4f513"

headers = {
    "Content-Type": "application/json",
    "X-DBP-APIKEY": API_KEY
}

def run_query(query):
    payload = {'query': query}
    response = requests.post(API_URL, json=payload, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Query failed with status code {response.status_code}: {response.text}")

### 2. Breakdown Logic Function

We define a robust, generic Python function `breakdown_min_lot_size(product_code)` which dynamically fetches and joins all data from the API to allocate the proper `MinLotSize` to individual contracts.

In [0]:
def breakdown_min_lot_size(product_code):
    """
    Performs a 3-step retrieval and join to allocate product-level TES block trade thresholds
    (MinLotSize) to individual contracts for a given product filter.
    """
    print(f"Processing MinLotSize breakdown for product: {product_code}...\n")
    
    # Step 1: Fetch Expirations
    exp_query = f"""
    query {{
      Expirations(filter: {{ Product: {{ eq: \"{product_code}\" }} }}) {{
        data {{
          ProductID
          Product
          MasterContract
          ExpirationIndex
          ExpirationDate
        }}
      }}
    }}
    """
    exp_res = run_query(exp_query)
    exp_data = exp_res.get("data", {}).get("Expirations", {}).get("data", [])
    if not exp_data:
        print(f"No expirations found for product {product_code}.")
        return pd.DataFrame()
    df_exp = pd.DataFrame(exp_data)
    
    # Step 2: Fetch TESProfiles for InstrumentType = SIMPLE_INSTRUMENT and TESType = BLOCK
    tes_query = f"""
    query {{
      TESProfiles(filter: {{ 
        Product: {{ eq: \"{product_code}\" }}, 
        InstrumentType: {{ eq: \"SIMPLE_INSTRUMENT\" }}, 
        TESType: {{ eq: \"BLOCK\" }} 
      }}) {{
        data {{ 
          ProductID
          Product
          InstrumentType
          TESType
          MinLotSize
          NonDisclosureLimit
          MinExpiryRange
        }}
      }}
    }}
    """
    tes_res = run_query(tes_query)
    tes_data = tes_res.get("data", {}).get("TESProfiles", {}).get("data", [])
    if not tes_data:
        print(f"No BLOCK TES profiles found for product {product_code}.")
        return pd.DataFrame()
    df_tes = pd.DataFrame(tes_data)
    
    # Step 3: Fetch Contracts
    contracts_query = f"""
    query {{
      Contracts(filter: {{ Product: {{ eq: \"{product_code}\" }} }}) {{
        data {{
          ProductID
          Product
          MasterContract
          ContractID
          Contract
          ExpirationDate
          ISIN
        }}
      }}
    }}
    """
    contracts_res = run_query(contracts_query)
    contracts_data = contracts_res.get("data", {}).get("Contracts", {}).get("data", [])
    if not contracts_data:
        print(f"No contracts found for product {product_code}.")
        return pd.DataFrame()
    df_contracts = pd.DataFrame(contracts_data)
    
    # Align: Join Contracts with Expirations on ProductID and MasterContract
    df_merged = pd.merge(
        df_contracts, 
        df_exp,
        on=["ProductID", "MasterContract"],
        suffixes=("", "_exp")
    )
    
    # Keep relevant contract columns
    df_merged = df_merged[[
        "ProductID", "Product", "MasterContract", "ContractID", 
        "Contract", "ExpirationDate", "ExpirationIndex", "ISIN"
    ]]
    
    # Allocate: Map ExpirationIndex to MinLotSize using sorted TESProfiles
    # MinExpiryRange indicates the START of a range (not the upper limit)
    # Sort descending to find the applicable range from highest to lowest
    df_tes_sorted = df_tes.sort_values("MinExpiryRange", ascending=False).copy()
    
    def find_threshold_value(exp_idx, df_profiles, field_name):
        # Find the first profile where ExpirationIndex >= MinExpiryRange
        # (profiles are sorted descending, so we find the highest applicable threshold)
        valid_profiles = df_profiles[exp_idx >= df_profiles["MinExpiryRange"]]
        if not valid_profiles.empty:
            # Return the field value for the first valid range (highest threshold that applies)
            return valid_profiles.iloc[0][field_name]
        else:
            # Should not happen if data is complete, but fallback to first profile
            return df_profiles.iloc[-1][field_name] if not df_profiles.empty else None
            
    df_merged["MinLotSize"] = df_merged["ExpirationIndex"].apply(
        lambda idx: find_threshold_value(idx, df_tes_sorted, "MinLotSize")
    )
    df_merged["NonDisclosureLimit"] = df_merged["ExpirationIndex"].apply(
        lambda idx: find_threshold_value(idx, df_tes_sorted, "NonDisclosureLimit")
    )
    
    return df_merged

### 3. Execution & Results

Let's execute this logic for product **IBE** (Iberdrola). This product has multiple expirations spanning several years and different `MinLotSize` tiers (e.g., `1500` for near-term expirations and `1` for far-term expirations).

We will print the final joined result displaying the mapped `MinLotSize` alongside each individual contract.

In [0]:
df_result = breakdown_min_lot_size("OESX")

if not df_result.empty:
    print(f"Total contracts processed: {len(df_result)}")
    
    # Sort contracts chronologically by ExpirationIndex for a clear, structured view
    df_result = df_result.sort_values(["ExpirationIndex", "ContractID"])
    
    # Get one representative contract per ExpirationIndex to show the mapping
    example_records = df_result.drop_duplicates(subset=["ExpirationIndex"])
    
    print("\n--- Example Records showing ExpirationIndex mapping to MinLotSize ---")
    display(example_records[["Product", "ContractID", "Contract", "ExpirationIndex", "MinLotSize", "NonDisclosureLimit", "ISIN"]])
else:
    print("Could not process breakdown.")